# Airbnb NYC Price Prediction Workflow

Complete, step-by-step notebook for loading the Airbnb NYC dataset, cleaning it, training baseline regression models, and visualizing performance to support future model improvements.

## 1. Imports & Configuration
Set up libraries used throughout the notebook. Adjust plotting style for consistency.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
DATA_PATH = Path('AB_NYC_2019.csv')
print(f"Using dataset at: {DATA_PATH.resolve()}")
CV_FOLDS = 3

## 2. Load & Inspect Data
Preview the raw dataset to understand available columns, data types, and missing values.

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
print(f"Original shape: {raw_df.shape}")
raw_df.head()

In [ ]:
print('Column overview:')
print(raw_df.dtypes)
print('
Missing values per column:')
print(raw_df.isna().sum())

## 3. Data Cleaning
- Fill `reviews_per_month` with 0 since missing implies no recent reviews.
- Drop rows missing critical categorical info (`neighbourhood_group`, `room_type`).
- Keep only positive prices and trim the most extreme 1% tails to reduce noise.

In [ ]:
clean_df = raw_df.copy()
clean_df['reviews_per_month'] = clean_df['reviews_per_month'].fillna(0)
clean_df = clean_df.dropna(subset=['neighbourhood_group', 'room_type'])
clean_df = clean_df[clean_df['price'] > 0]
lower = clean_df['price'].quantile(0.01)
upper = clean_df['price'].quantile(0.99)
clean_df = clean_df[(clean_df['price'] >= lower) & (clean_df['price'] <= upper)]
print(f"Cleaned shape: {clean_df.shape}")
clean_df.head()

## 4. Feature & Target Selection
Use geographic, operational, and categorical attributes to predict nightly price.

In [ ]:
feature_cols = [
    'latitude',
    'longitude',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365',
    'neighbourhood_group',
    'room_type',
]

target_col = 'price'
X = clean_df[feature_cols]
y = clean_df[target_col]
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

## 5. Train/Test Split
Reserve 20% of the data for unbiased evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

## 6. Preprocessing & Model Definitions
Standardize numeric features, one-hot encode categoricals, then configure linear, distance-based, and tree-based regressors inside identical pipelines. We'll also set up k-fold cross-validation (CV) to estimate generalization performance.

In [ ]:
numeric_features = [
    'latitude',
    'longitude',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365',
]

categorical_features = ['neighbourhood_group', 'room_type']

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1, max_iter=10000),
    'KNN Regression (k=5)': KNeighborsRegressor(n_neighbors=5),
    'Decision Tree': DecisionTreeRegressor(max_depth=None, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
}

models

## 7. Train & Evaluate Models
Fit each pipeline, compute train/test Mean Squared Error, and compare results.

In [ ]:
results = []
trained_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator),
    ])

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=CV_FOLDS,
        scoring='neg_mean_squared_error',
        n_jobs=1,
    )
    cv_mse = (-cv_scores).mean()

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'Train MSE': mean_squared_error(y_train, train_pred),
        'Test MSE': mean_squared_error(y_test, test_pred),
        f'CV MSE ({CV_FOLDS}-fold)': cv_mse,
    })

results_df = pd.DataFrame(results).sort_values('Test MSE').reset_index(drop=True)
results_df

In [ ]:
best_model_name = results_df.loc[0, 'Model']
print(f"Best Test MSE model: {best_model_name}")

## 8. Visualizations
1. Bar chart comparing Train/Test MSE.
2. Scatter plot of actual vs. predicted prices for the best model.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(results_df))
ax.bar(x_pos - 0.15, results_df['Train MSE'], width=0.3, label='Train MSE')
ax.bar(x_pos + 0.15, results_df['Test MSE'], width=0.3, label='Test MSE')
ax.set_xticks(x_pos)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Model Comparison')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_model = trained_models[best_model_name]
best_pred = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_pred, alpha=0.5)
max_val = max(y_test.max(), best_pred.max())
plt.plot([0, max_val], [0, max_val], 'r--', label='Ideal fit')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Actual vs Predicted Prices ({best_model_name})')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Next Steps
- Tune hyperparameters (e.g., Ridge/Lasso alphas, KNN neighbors, tree depth, forest size, learning rate) via GridSearchCV/RandomizedSearchCV.
- Engineer richer features (text sentiment, neighbourhood-level stats, temporal signals) and re-train the expanded model set.
- Explore advanced ensembles (e.g., XGBoost/LightGBM) or stacking models to push Test MSE lower once strong single models are identified.